# LLM Training & Tokenization: From Text to Tensors

## What are we trying to understand?

This notebook uses a practical question to organize the material:

> **Which tools and approaches are useful for training or adapting LLMs, and how do tokens, embeddings, vectors, and tensors move through that process?**

The notebook is intentionally **neutral**. It does not assume that PyTorch, TensorFlow, a particular tokenizer, or a particular fine-tuning method is universally best.

Instead, we will:

- inspect tokenization directly
- compare several tokenization strategies and their use cases
- connect token IDs to embeddings and tensors
- distinguish LLM pretraining, fine-tuning, RAG, and inference
- compare the roles of PyTorch, TensorFlow, Hugging Face, PEFT/LoRA, MLflow, vLLM, LangChain, and LangGraph
- identify the measurements that matter when evaluating an LLM training approach

### A useful mental model

```text
Text
  ↓
Tokenizer
  ↓
Token IDs
  ↓
Embedding lookup
  ↓
Vectors / tensors
  ↓
Transformer
  ↓
Logits / next-token probabilities
  ↓
Training or inference
```

**Important:** token IDs are not universal numbers. They are indices in the vocabulary of a particular tokenizer/model.

## 1. The major tools and where they fit

| Tool | Primary role | Train LLMs? | Fine-tune? | Inference? | Application layer? |
|---|---|---:|---:|---:|---:|
| **NumPy** | Numerical arrays | Indirect | Indirect | Supporting | No |
| **scikit-learn** | Classical ML | Generally no | Generally no | Yes | No |
| **PyTorch** | Deep learning | **Yes** | **Yes** | **Yes** | No |
| **TensorFlow/Keras** | Deep learning | **Yes** | **Yes** | **Yes** | No |
| **Hugging Face Transformers** | LLM models/tokenizers/training ecosystem | **Yes** | **Yes** | **Yes** | Sometimes |
| **PEFT / LoRA** | Parameter-efficient adaptation | No | **Yes** | Yes | No |
| **MLflow** | Experiment/model tracking | No | No | Supporting | No |
| **vLLM** | LLM inference/serving | No | No | **Yes** | No |
| **LangChain** | LLM application integration | No | No | Through models | **Yes** |
| **LangGraph** | Stateful agent orchestration | No | No | Through models | **Yes** |
| **Pydantic** | Data validation/schema contracts | No | No | Supporting | **Yes** |

### The key distinction

These are not all competitors.

For example:

```text
PyTorch / TensorFlow
        ↓
   model training

Hugging Face
        ↓
models + tokenizers + datasets + training ecosystem

PEFT / LoRA
        ↓
efficient adaptation of an existing model

vLLM
        ↓
high-throughput inference

LangChain / LangGraph
        ↓
applications and agent workflows
```

The notebook will compare technologies **according to the job they perform**.

# 2. Tokenization: the first transformation

An LLM does not receive a sentence as a Python string.

A typical pipeline is:

```text
"The cat sat."
      ↓
   tokenizer
      ↓
["The", " cat", " sat", "."]
      ↓
[464, 3797, 3332, 13]
      ↓
embedding lookup
```

The exact tokens and IDs depend on the tokenizer.

## Token IDs are lookup indices

There is normally **no mathematical formula that converts a word into a token ID**.

Instead:

```text
tokenizer vocabulary
       │
       ├── token → ID
       │
       └── ID → token
```

The vocabulary was constructed as part of the tokenizer/model ecosystem.

Therefore:

```text
"cat" → 3797
```

might be true for one tokenizer and completely different for another.

It might also be split:

```text
"unbelievable"
       ↓
["un", "believ", "able"]
```

rather than being one token.

In [ ]:
# Lightweight tokenization example without requiring a model download.
# This illustrates the concept, not a production tokenizer.

text = "The cat sat on the mat."

words = text.split()
characters = list(text.replace(" ", ""))

print("Text:", text)
print("\nWord-level:")
print(words)

print("\nCharacter-level:")
print(characters[:30], "...")

print("\nNotice:")
print("- Word tokenization produces fewer units.")
print("- Character tokenization produces many more units.")
print("- Modern LLMs generally use subword/token-piece strategies.")

# 3. Tokenization methods and when they make sense

There is no universally best tokenizer. The appropriate choice depends on the model, training corpus, language mix, code/text mix, context length, and objective.

## Word-level tokenization

```text
"Machine learning is useful"
        ↓
["Machine", "learning", "is", "useful"]
```

**Useful for:** teaching, simple NLP pipelines, early/classical NLP.

**Problem:** vocabulary becomes enormous and previously unseen words are difficult to represent efficiently.

---

## Character-level tokenization

```text
"cat"
 ↓
["c", "a", "t"]
```

**Useful for:** situations where spelling, morphology, noisy text, or arbitrary character sequences matter.

**Problem:** sequences become much longer, which can make transformer computation expensive.

---

## WordPiece

A word can be represented using reusable pieces:

```text
"playing"
   ↓
["play", "##ing"]
```

**Common association:** BERT-style language models.

**Useful when:** a model benefits from subword representations that handle previously unseen words through smaller pieces.

---

## Byte Pair Encoding (BPE)

BPE builds a vocabulary of frequently occurring pieces.

A conceptual example:

```text
"playing"
   ↓
["play", "ing"]
```

**Common association:** GPT-family and other modern language-model tokenizers.

**Useful when:** balancing vocabulary size against sequence length for general text/code workloads.

---

## SentencePiece

SentencePiece performs tokenization without requiring conventional whitespace-separated words as its starting assumption.

**Useful when:**

- multilingual text matters
- whitespace conventions differ
- you want a tokenizer that operates directly on raw text
- a model's training ecosystem was built around SentencePiece

SentencePiece can implement different segmentation approaches, including BPE and Unigram.

---

## Unigram

Unigram tokenization considers alternative segmentations and selects pieces according to a probabilistic model.

**Useful when:** the tokenizer/model ecosystem is designed around flexible probabilistic subword segmentation.

### The important engineering point

The name of the algorithm alone does not tell you which tokenizer is best.

You need to measure it against **your corpus and model objective**.

# 4. Same text, different token counts

Tokenization affects how much text fits into a context window.

Imagine a document containing 100,000 words:

```text
Tokenizer A → 130,000 tokens
Tokenizer B → 115,000 tokens
Tokenizer C → 160,000 tokens
```

If the model context limit is 128,000 tokens:

- A does not fit as one sequence.
- B fits.
- C does not fit.

That affects:

- training sequence construction
- batching
- memory requirements
- context utilization
- RAG chunking
- inference cost
- how much information can fit into a single context

For this reason, **token efficiency is one useful measurement when evaluating tokenizers**.

But fewer tokens are not automatically better. Token boundaries also affect how effectively the model represents the underlying language or code.

In [ ]:
# A simple token-efficiency calculation.
# This is independent of any particular tokenizer.

def token_efficiency(text, token_count):
    characters = len(text)
    return {
        "characters": characters,
        "tokens": token_count,
        "characters_per_token": characters / token_count,
        "tokens_per_1000_characters": token_count / characters * 1000,
    }

sample = "Machine learning systems process text using tokenization."

print(token_efficiency(sample, token_count=10))

# 5. Real tokenizer experiment with Hugging Face

The following experiment uses a pretrained tokenizer.

If the package/model is not already installed or cached, the first run may need internet access.

We will inspect:

1. the actual token pieces
2. their token IDs
3. the decoded pieces
4. the number of tokens

The **same text can produce different IDs and token counts with different tokenizers**.

In [ ]:
# Uncomment if needed:
# %pip install -q transformers

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "The cat sat on the mat."

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, add_special_tokens=False)

print("Text:", text)
print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nToken count:", len(token_ids))

print("\nToken → ID")
for token, token_id in zip(tokens, token_ids):
    print(f"{token!r:15} → {token_id}")

In [ ]:
# Reverse the lookup: ID → token/text

print("ID → decoded token")
for token_id in token_ids:
    print(f"{token_id:5} → {tokenizer.decode([token_id])!r}")

# 6. Why whitespace and subwords matter

Notice that some tokenizers encode whitespace as part of a token.

For example, conceptually:

```text
"cat"
" cat"
```

may have different tokenization.

Likewise, a rare or unfamiliar word may be split:

```text
"electrification"
        ↓
["elect", "rification"]
```

while a common word may have a single token.

This is one reason token IDs should **never be interpreted as semantic numbers**.

Token ID 500 is not inherently “more similar” to token ID 501 than to token ID 10,000.

The semantic representation emerges later through the learned embedding/model parameters.

# 7. Token IDs → embeddings → tensors

This is the bridge between tokenization and neural-network computation.

Suppose:

```text
Token IDs:

[464, 3797, 3332]
```

The model has an embedding matrix:

```text
vocabulary_size × embedding_dimension
```

For example:

```text
50,000 × 768
```

A token ID selects one row:

```text
ID 464
  ↓
embedding matrix[464]
  ↓
768-dimensional vector
```

For a sequence:

```text
3 token IDs
      ↓
3 embedding vectors
      ↓
tensor shape = (3, 768)
```

For a batch:

```text
batch_size = 8
sequence_length = 3
embedding_dimension = 768

tensor shape = (8, 3, 768)
```

This is where **tokens become numerical tensors that the neural network can process**.

In [ ]:
import numpy as np

vocabulary_size = 50_000
embedding_dimension = 8

# A tiny simulated embedding matrix.
embedding_matrix = np.random.randn(
    vocabulary_size,
    embedding_dimension
).astype(np.float32)

token_ids = np.array([464, 3797, 3332])

embeddings = embedding_matrix[token_ids]

print("Token IDs shape:", token_ids.shape)
print("Embedding shape:", embeddings.shape)
print("\nEmbedding vectors:")
print(embeddings)

# 8. What happens inside an LLM?

A simplified autoregressive training path looks like:

```text
Raw text
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Embedding lookup
   ↓
Token embeddings
   +
Positional information
   ↓
Transformer blocks
   ↓
Hidden-state tensors
   ↓
Linear projection
   ↓
Logits
   ↓
Next-token probabilities
   ↓
Loss
   ↓
Backpropagation
   ↓
Parameter updates
```

During training, the model repeatedly tries to predict the next token.

For example:

```text
Input:
"The cat sat on the"

Target:
"mat"
```

The model produces probabilities over its vocabulary, and the training loss measures how well it predicted the target.

# 9. Training vs fine-tuning vs RAG vs inference

These are often mixed together, but they are different operations.

### Pretraining

```text
Huge corpus
    ↓
Tokenizer
    ↓
LLM
    ↓
next-token prediction
    ↓
billions/trillions of training examples
```

Goal: learn broad language/model representations.

### Fine-tuning

```text
Existing pretrained model
          ↓
       new dataset
          ↓
   parameter updates
          ↓
      adapted model
```

Goal: adapt the model to a task/domain/style.

### PEFT / LoRA

```text
Existing model
     │
     ├── mostly frozen
     │
     └── small trainable adapters
              ↓
        adapted model
```

Goal: reduce the number of trainable parameters and resource requirements.

### RAG

```text
Documents
   ↓
Embedding model
   ↓
Vectors
   ↓
Vector database
   ↓
Relevant context
   ↓
LLM
```

**RAG does not necessarily train the LLM.**

### Inference

```text
Prompt
  ↓
Tokenizer
  ↓
LLM
  ↓
next-token generation
  ↓
response
```

This distinction is critical when comparing tools.

# 10. Where the major frameworks fit

## PyTorch

The core deep-learning framework.

```text
Tensor
  ↓
Neural network
  ↓
Loss
  ↓
Gradient
  ↓
Optimizer
  ↓
Updated parameters
```

**Relevant to:** training from scratch, fine-tuning, research, custom architectures, inference.

---

## TensorFlow / Keras

Another complete deep-learning ecosystem.

```text
Tensor
  ↓
Keras model
  ↓
Loss
  ↓
Gradient/update
```

**Relevant to:** neural-network training, fine-tuning, deployment, production ML.

---

## Hugging Face Transformers

Provides a major ecosystem around pretrained transformer models, tokenizers, datasets, and training workflows.

**Relevant to:** practical LLM experimentation, fine-tuning, model loading, tokenization and evaluation.

---

## PEFT / LoRA

Focuses on efficient adaptation of existing models.

**Relevant to:** fine-tuning when full parameter updates are unnecessary or too expensive.

---

## MLflow

Tracks experiments, metrics, parameters, artifacts and models.

**Relevant to:** reproducibility and comparing training experiments.

---

## vLLM

Optimizes LLM serving/inference.

**Relevant to:** high-throughput generation after a model has been trained or fine-tuned.

---

## LangChain / LangGraph

These sit further up the stack.

**LangChain:** connect models to tools, retrievers and application components.

**LangGraph:** represent stateful, multi-step agent workflows.

Neither is a replacement for PyTorch or TensorFlow for LLM training.

# 11. How to compare LLM training approaches fairly

Instead of assigning a subjective "best" score, compare measurable characteristics.

## Training efficiency

- tokens processed per second
- training time
- GPU utilization
- GPU memory usage
- number of trainable parameters
- batch size
- sequence length

## Model efficiency

- parameter count
- model size
- tokenization efficiency
- inference latency
- tokens/second
- memory required for inference

## Quality

Depending on the task:

- validation loss
- perplexity
- task-specific accuracy
- instruction-following evaluations
- retrieval quality
- human evaluation

## Engineering

- ease of debugging
- ecosystem maturity
- pretrained model availability
- distributed-training support
- checkpointing
- experiment tracking
- deployment options

### Important

A framework can be excellent for one objective and poorly suited to another.

For example:

```text
Train from scratch
       ≠
Fine-tune a 7B model
       ≠
RAG
       ≠
Serve a 70B model
       ≠
Build an agent
```

The notebook therefore compares **workflows and measurable outcomes**, not just library names.

# 12. A practical decision map

```text
What are you trying to do?
          │
          ├── Train a new neural model
          │       ↓
          │   PyTorch / TensorFlow
          │
          ├── Fine-tune an existing LLM
          │       ↓
          │   Hugging Face
          │       +
          │   PyTorch / TensorFlow
          │       +
          │   PEFT / LoRA when appropriate
          │
          ├── Add external knowledge without changing model weights
          │       ↓
          │   Embeddings + vector retrieval + RAG
          │
          ├── Serve a trained LLM
          │       ↓
          │   vLLM or another inference stack
          │
          └── Build an agent/application
                  ↓
              LangChain / LangGraph
```

This is the central perspective for the rest of the lab: **follow the data and tensors through the system, then compare the tools at the point where each actually contributes.**

# 13. Suggested next experiments

The next useful experiments for this notebook are:

1. **Compare GPT-2, BERT and a SentencePiece-based tokenizer** on the same text.
2. Measure token counts for **English, source code, and another language**.
3. Visualize a small embedding matrix.
4. Run a tiny PyTorch transformer training loop.
5. Repeat the conceptual training loop with TensorFlow/Keras.
6. Fine-tune a small pretrained model with Hugging Face.
7. Compare full fine-tuning with LoRA.
8. Track experiments with MLflow.
9. Send the resulting model through an inference server.
10. Build a tiny RAG pipeline and demonstrate why **embeddings/vectors are not the same thing as training the LLM**.

Those experiments would turn this from a reference notebook into an actual **LLM framework comparison lab**.